In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import numpy as np

In [ ]:
def process_df(df, feature_code):
    # feature_code: [old/new/no]-pol_[old/new/no]-pi_[y/n]-ecfp_[all/no/x/y/z]-torsion_[all/no/x/y/z]-sd
    if feature_code.endswith('_pt'):
        feature_code = feature_code.split('_pt')[0]
        first_cols = ['Monomer 1', 'Monomer 2']
    else:
        first_cols = ['Monomer 1', 'Monomer 2', 'Oligomer Type']
        
    pol, pi, ecfp = feature_code.split('_')[:-2]
    torsion_selection = feature_code.split('_')[-2]
    sd_selection = feature_code.split('_')[-1]
    
    geo_options = ['x', 'y', 'z']
    torsion_options = ['Total Torsion Angle', 'Mean Torsion Angle', 'Inner Mean Torsion Angle']
    sd_options = ['SD Torsions', 'SD Difference']
    
    geo_select_dict = {
        'all-torsion': [f"{t_opt}_{g_opt}" for t_opt in torsion_options for g_opt in geo_options],
        'all-sd': [f"{s_opt}_{g_opt}" for s_opt in sd_options for g_opt in geo_options],
        'no-torsion': [],
        'no-sd': [],
        'x-torsion': [f"{t_opt}_x" for t_opt in torsion_options],
        'y-torsion': [f"{t_opt}_y" for t_opt in torsion_options],
        'z-torsion': [f"{t_opt}_z" for t_opt in torsion_options],
        'x-sd': [f"{s_opt}_x" for s_opt in sd_options],
        'y-sd': [f"{s_opt}_y" for s_opt in sd_options],
        'z-sd': [f"{s_opt}_z" for s_opt in sd_options],
    }
    
    others_select_dict = {
        'old-pol': ['Molecule Polarizability', 'Average Monomer Polarizability', 'Dipole Moment'],
        'new-pol': ['Rotated Polarizability', 'Average Monomer Polarizability', 'Dipole Moment', 'α_xx', 'α_xy', 'α_yy'],
        'old-pi': ['PiSystemSize'],
        'new-pi': ['PiSystemSize', 'PiSystemArea', 'PiSystemLength'],
        'y-ecfp': [f"ECFP_{i}" for i in range(4096)],
        'no-pi': [],
        'no-pol': [],
        'n-ecfp': []
    }
    
    pol_cols = others_select_dict.get(pol)
    torsion_cols = geo_select_dict.get(torsion_selection)
    sd_cols = geo_select_dict.get(sd_selection)
    pi_cols = others_select_dict.get(pi)
    ecfp_cols = others_select_dict.get(ecfp)
    spec_cols = ['DFT Oscillator Strength', 'DFT CD Intensity']
    
    if pol.split('-')[0] != 'no' and pi.split('-')[0] != 'no':
        energy_cols = [
            'Molecule HOMO (eV)', 'Molecule LUMO (eV)', 'Molecule HLG (eV)', 
            'ΔE_HOMO (eV)', 'ΔE_LUMO (eV)', 'Monomer HLG (eV)']
        first_cols.extend(energy_cols)

    col_order = first_cols + pol_cols + torsion_cols + sd_cols + pi_cols + ecfp_cols + spec_cols
    
    df = df[col_order]
    
    df = df.rename(columns = {
        'DFT CD Intensity': 'DFT CD Intensity (a.u.)',
        'DFT Oscillator Strength': 'DFT Oscillator Strength (a.u.)',
    })
    
    new_col_names = []
    for col in df.columns:
        if 'Torsion' in col or 'SD' in col:
            new_col_names.append(col + ' (°)')
        elif 'Polarizability' in col or 'α' in col:
            new_col_names.append(col + ' (Å$^3$)')
        elif 'Dipole' in col:
            new_col_names.append(col + ' (Debye)')
        else:
            new_col_names.append(col)
    
    df.columns = new_col_names
    
    if len(torsion_selection.split('-')[0]) < 2:
        df = df.rename(columns = {
            'Total Torsion Angle_z (°)': 'Total Torsion Angle (°)',
            'Mean Torsion Angle_z (°)': 'Mean Torsion Angle (°)',
            'Inner Mean Torsion Angle_z (°)': 'Inner Mean Torsion Angle (°)',
        })
    
    if len(sd_selection.split('-')[0]) < 2:
        df = df.rename(columns = {
            'SD Torsions_x (°)': 'SD Torsions (°)',
            'SD Difference_x (°)': 'SD Difference (°)'
        })
    
    df = df.dropna().reset_index(drop = True)
    
    print(f'Number of features (m): {len(df.columns)}')
    print(f'Number of data points (n): {len(df)}')
    print(f"Input features: {list(dict.fromkeys([x.split('_')[0] for x in df.drop(['DFT CD Intensity (a.u.)'], axis = 1).columns]))}\n")
    return df

In [ ]:
def process_corr(corr_df, corr_method, output_path, output):
    cd_row = corr_df[['DFT CD Intensity (a.u.)']].drop('DFT CD Intensity (a.u.)').reset_index()
    cd_row = cd_row.rename(columns = {'index': 'Feature', 'DFT CD Intensity (a.u.)': 'Correlation'})
    cd_row['Abs. Correlation'] = cd_row['Correlation'].abs()
    cd_row = cd_row.sort_values(by = 'Abs. Correlation', ascending = False).reset_index(drop = True)
    cd_row.to_csv(os.path.join(output_path, f"{output}_{corr_method}_scores.csv"), index = False)

    return cd_row

In [ ]:
def make_heatmap(df, corr_method, output_path, output, preview):
    fig = plt.figure(figsize = (18,12), facecolor = 'w', edgecolor = 'k')
    sns.heatmap(df, annot = True)
    plt.xticks(rotation = 30, ha = 'right', fontsize = 14)
    plt.yticks(fontsize = 14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{output}_{corr_method}_heatmap.pdf"))
    #plt.savefig(os.path.join(output_path, f"{output}.png"), dpi = 300)
    
    if preview == False:
        plt.close()

In [ ]:
def make_barchart(df, corr_method, my_cmap, output_path, output, preview):
    heatmap_features = df['Feature'].tolist()
    heatmap_coeffs = df['Abs. Correlation'].tolist()
    indices = np.argsort(heatmap_coeffs)
    num_features = min(10, len(heatmap_coeffs)) 

    my_cmap = plt.get_cmap(name=custom_cmap10)
    fig, ax = plt.subplots(figsize=(10, 6))

    selected_indices = indices[-num_features:][::-1]

    selected_coeffs = [heatmap_coeffs[i] for i in selected_indices][::-1]
    selected_features = [heatmap_features[i] for i in selected_indices][::-1]
    
    if output.endswith('aa'):
        for i in range(len(selected_features)):
            if '_x (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_x (°)', '_0 (°)')
            elif '_y (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_y (°)', '_1 (°)')
            elif '_z (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_z (°)', '_2 (°)')
    
    colors = [my_cmap(i / num_features) for i in reversed(range(num_features))]

    ax.barh(range(num_features), selected_coeffs, color=colors, edgecolor = 'k', align='center')
    plt.yticks(range(num_features), selected_features, fontsize=16)
    plt.xlabel("|Pearson's r|", fontsize=20)
    plt.xticks(fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{output}_{corr_method}_hbar.pdf"))
    #plt.savefig(os.path.join(output_path, f"{output}_hbar.png"), dpi = 300)
    
    if preview == False:
        plt.close()

In [ ]:
def main(df, ft_code, corr_method, my_cmap, output_path, preview):
    os.makedirs(output_path, exist_ok = True)
    
    output_name = ft_dict.get(ft_code)
    if output_name.endswith('_pt'):
        df = df[df['Oligomer Type'] == 2].reset_index(drop = True)
    
    data_df = df.copy(deep = True)
    data_df = process_df(data_df, ft_code)
    
    corr_df = data_df.corr(method = corr_method)
    cd_row = process_corr(corr_df, corr_method, output_path, output_name)
    make_heatmap(corr_df, corr_method, output_path, output_name, preview = False)
    make_barchart(cd_row, corr_method, my_cmap, output_path, output_name, preview = False)

In [ ]:
color_palette_10 = ['#24E4BC', '#00CFF9', '#648FFF', '#785EF0', '#AA42B8', '#DC267F', '#ED2F2C', '#FE6100', '#FFB000', '#FFE200']
custom_cmap10 = ListedColormap(color_palette_10, name = 'cb10')

ft_dict = {
    'old-pol_old-pi_n-ecfp_all-torsion_all-sd': 'oonaa',
    'old-pol_old-pi_n-ecfp_z-torsion_x-sd': 'oonzx',
    'old-pol_old-pi_n-ecfp_z-torsion_x-sd_pt': 'oonzx_pt',
    'new-pol_old-pi_n-ecfp_z-torsion_x-sd': 'nonzx',
    'new-pol_new-pi_n-ecfp_z-torsion_x-sd': 'nnnzx',
}

ft_codes = list(ft_dict.keys())

df = pd.read_csv('clean_dft_df_4096.csv')

corr_method = 'pearson'
output_path = os.path.join(os.getcwd(), 'output', corr_method)

In [ ]:
preview = False

In [ ]:
for i in range(len(ft_codes)):
    ft_code = ft_codes[i]
    main(df, ft_code, corr_method, custom_cmap10, output_path, preview)